# Problem: Write Sequence-to-Sequence with Attention

### Problem Statement
Implement a **Sequence-to-Sequence (Seq2Seq) model with Attention** by completing the required sections. The model consists of an **Encoder** that processes input sequences and a **Decoder** with an attention mechanism that generates output sequences.

### Requirements

1. **Encoder Class**:
   - **Layers**:
     - Use an embedding layer to map input tokens to dense vectors.
     - Use an LSTM layer to capture temporal dependencies in the sequence.
   - **Forward Pass**:
     - Pass the input sequence through the embedding layer.
     - Feed the embedded sequence into the LSTM.
     - Return the LSTM outputs and the final hidden and cell states.

2. **Decoder with Attention**:
   - **Layers**:
     - Use an embedding layer to process output sequence tokens.
     - Implement an attention mechanism to compute attention weights between the encoder outputs and the current decoder hidden state.
     - Use an LSTM layer to predict the next token using the context vector (from attention) and the current decoder state.
     - Use a fully connected output layer to predict the next token.
   - **Forward Pass**:
     - Process the input through the embedding layer.
     - Compute attention weights using the decoder hidden state and encoder outputs.
     - Calculate the context vector by applying the attention weights to the encoder outputs.
     - Combine the context vector with the embedded input.
     - Feed the combined representation into the LSTM.
     - Pass the LSTM output through a fully connected layer to predict the next token.


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

In [22]:
def lstm_loop(W, U, x, ch0, reverse=False):
    B, T, embed_dim = x.shape
    c, h = ch0
    
    maybe_reverse = (lambda x: tuple(reversed(x))) if reverse else (lambda x: x)
    
    hh = []
    for t in maybe_reverse(range(T)):
        f, i, o, e = (W(h) + U(x[:,t,:])).chunk(4, dim=-1)
        f = torch.sigmoid(f)
        i = torch.sigmoid(i)
        o = torch.sigmoid(o)
        e = torch.tanh(e)
        c = f * c + i * e
        h = o * torch.tanh(c)
        hh.append(h)

    return torch.stack(maybe_reverse(hh), dim=1), (c, h)

class LSTMLayer(nn.Module):
    def __init__(self, input_dim, embed_dim, bidirectional=False):
        super(LSTMLayer, self).__init__()
        self.input_dim = input_dim
        self.embed_dim = embed_dim
        self.bidirectional = bidirectional
        
        self.Wf = nn.Linear(embed_dim, embed_dim*4)
        self.Uf = nn.Linear(input_dim, embed_dim*4)
        
        if bidirectional:
            self.Wb = nn.Linear(embed_dim, embed_dim*4)
            self.Ub = nn.Linear(input_dim, embed_dim*4)

            self.Po = nn.Linear(embed_dim*2, embed_dim)
            self.Pc = nn.Linear(embed_dim*2, embed_dim)
            self.Ph = nn.Linear(embed_dim*2, embed_dim)
            
    def forward(self, x, ch0=None):
        B, T, input_dim = x.shape
        
        if self.bidirectional:
            if ch0 is None:
                c0 = torch.zeros(B, self.embed_dim*2)
                h0 = torch.zeros(B, self.embed_dim*2)
            else:
                c0, h0 = ch0

            c0f, c0b = c0.chunk(2, dim=-1)
            h0f, h0b = h0.chunk(2, dim=-1)
            ch0f, ch0b = (c0f, h0f), (c0b, h0b)
            
            of, (cf, hf) = lstm_loop(self.Wf, self.Uf, x, ch0f, reverse=False)
            ob, (cb, hb) = lstm_loop(self.Wb, self.Ub, x, ch0b, reverse=True)
            
            o = self.Po(torch.cat((of, ob), dim=-1))
            c = self.Pc(torch.cat((cf, cb), dim=-1))
            h = self.Ph(torch.cat((hf, hb), dim=-1))
        else:
            if ch0 is None:
                c0 = torch.zeros(B, self.embed_dim)
                h0 = torch.zeros(B, self.embed_dim)
                ch0 = (c0, h0)
            o, (c, h) = lstm_loop(self.Wf, self.Uf, x, ch0, reverse=False)

        return o, (c, h)

In [23]:
def scaled_dot_product_attention(qry, key, val, attn_mask=None):
    #       qry: B, H, T, D
    #       key: B, H, S, D
    #       val: B, H, S, D
    # attn_mask: B, H, T, S
    #    output: B, H, T, D
    
    B, H, T, D = qry.shape
    _, _, S, _ = key.shape
    
    attn_logit = torch.einsum("bhtd,bhsd->bhts", qry, key) * (1/D)**0.5

    if attn_mask is not None:
        attn_bias = torch.zeros(B, H, T, S)
        attn_bias.masked_fill_(~attn_mask, -torch.inf)
        attn_logit += attn_bias

    attn_weight = torch.softmax(attn_logit, dim=-1)
    output = torch.einsum("bhts,bhsd->bhtd", attn_weight, val)

    return output

class SelfAttention(nn.Module):
    def __init__(self, tgt_dim, embed_dim, num_heads, is_causal=False):
        super(SelfAttention, self).__init__()
        assert embed_dim % num_heads == 0
        self.tgt_dim = tgt_dim
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.is_causal = is_causal
        self.qry_projection = nn.Linear(tgt_dim, embed_dim)
        self.key_projection = nn.Linear(tgt_dim, embed_dim)
        self.val_projection = nn.Linear(tgt_dim, embed_dim)
        self.out_projection = nn.Linear(embed_dim, tgt_dim)

    def forward(self, x):
        B, T, tgt_dim = x.shape
        qry = self.qry_projection(x).reshape(B, T, self.num_heads, self.embed_dim // self.num_heads).permute(0, 2, 1, 3) # B, H, T, D
        key = self.key_projection(x).reshape(B, T, self.num_heads, self.embed_dim // self.num_heads).permute(0, 2, 1, 3) # B, H, T, D
        val = self.val_projection(x).reshape(B, T, self.num_heads, self.embed_dim // self.num_heads).permute(0, 2, 1, 3) # B, H, T, D

        if self.is_causal:
            mask = torch.ones(T, T, dtype=torch.bool).tril(diagonal=0).view(1, 1, T, T).expand(B, self.num_heads, T, T)  # B, H, T, S
        else:
            mask = None
        tmp = scaled_dot_product_attention(qry, key, val, attn_mask=mask)                                                # B, H, T, D
        tmp = tmp.permute(0, 2, 1, 3).reshape(B, T, self.embed_dim)

        return self.out_projection(tmp)

class CrossAttention(nn.Module):
    def __init__(self, tgt_dim, src_dim, embed_dim, num_heads):
        super(CrossAttention, self).__init__()
        assert embed_dim % num_heads == 0
        self.tgt_dim = tgt_dim
        self.src_dim = src_dim
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.qry_projection = nn.Linear(tgt_dim, embed_dim)
        self.key_projection = nn.Linear(src_dim, embed_dim)
        self.val_projection = nn.Linear(src_dim, embed_dim)

        self.out_projection = nn.Linear(embed_dim, tgt_dim)

    def forward(self, x, z):
        B, T, tgt_dim = x.shape
        B, S, src_dim = z.shape
        qry = self.qry_projection(x).reshape(B, T, self.num_heads, self.embed_dim // self.num_heads).permute(0, 2, 1, 3) # B, H, T, D
        key = self.key_projection(z).reshape(B, S, self.num_heads, self.embed_dim // self.num_heads).permute(0, 2, 1, 3) # B, H, S, D
        val = self.val_projection(z).reshape(B, S, self.num_heads, self.embed_dim // self.num_heads).permute(0, 2, 1, 3) # B, H, S, D

        tmp = scaled_dot_product_attention(qry, key, val)
        tmp = tmp.permute(0, 2, 1, 3).reshape(B, T, self.embed_dim)

        return self.out_projection(tmp)

In [35]:
class EncoderLayer(nn.Module):
    def __init__(self, embed_dim):
        super(EncoderLayer, self).__init__()
        self.auto_regression = LSTMLayer(embed_dim, embed_dim, bidirectional=True)
        self.fc_block = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4),
            nn.ReLU(),
            nn.Linear(embed_dim*4, embed_dim),
        )
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.layer_norm2 = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        x_tmp, (c, h) = self.auto_regression(x)
        x = x + x_tmp
        x = self.layer_norm1(x)
        x = x + self.fc_block(x)
        x = self.layer_norm2(x)
        return x, (c, h)

class Encoder(nn.Module):
    def __init__(self, src_vocab, embed_dim, num_layers):
        super(Encoder, self).__init__()
        # Add layers for embedding and LSTM
        self.embedding = nn.Embedding(src_vocab, embed_dim)
        self.encoder_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.encoder_layers.append(EncoderLayer(embed_dim))
            
    def forward(self, x):
        # Define the forward pass for the encoder
        x = self.embedding(x) # B,T   -> B,T,D
        cc, hh = [], []
        for encoder_layer in self.encoder_layers:
            x, (c, h) = encoder_layer(x)
            cc.append(c)
            hh.append(h)
        return x, (cc, hh)


class DecoderLayer(nn.Module):
    def __init__(self, tgt_dim, src_dim):
        super(DecoderLayer, self).__init__()
        self.auto_regression = LSTMLayer(tgt_dim, tgt_dim, bidirectional=False)
        self.cross_attention = CrossAttention(tgt_dim, src_dim, 64, 1) 
        self.fc_block = nn.Sequential(
            nn.Linear(tgt_dim, tgt_dim*4),
            nn.ReLU(),
            nn.Linear(tgt_dim*4, tgt_dim),
        )
        self.layer_norm1 = nn.LayerNorm(tgt_dim)
        self.layer_norm2 = nn.LayerNorm(tgt_dim)
        self.layer_norm3 = nn.LayerNorm(tgt_dim)

    def forward(self, x, z, ch0=None):
        x_tmp, (c, h) = self.auto_regression(x, ch0)
        x = x + x_tmp
        x = self.layer_norm1(x)
        x = x + self.cross_attention(x, z)
        x = self.layer_norm2(x)
        x = x + self.fc_block(x)
        x = self.layer_norm3(x)
        return x, (c, h)


# Define the Decoder with Attention
#TODO: Implement the Decoder class with Attention
class Decoder(nn.Module):
    def __init__(self, tgt_vocab, tgt_dim, src_dim, num_layers):
        super(Decoder, self).__init__()
        # Add layers for embedding, attention, LSTM, and output
        self.embedding = nn.Embedding(tgt_vocab, tgt_dim)
        self.decoder_layers = nn.ModuleList()
        for i in range(num_layers):
            self.decoder_layers.append(DecoderLayer(tgt_dim, src_dim))
        self.out_projection = nn.Linear(tgt_dim, tgt_vocab)

    def forward(self, x, encoder_outputs, cchh0):
        # Define the forward pass for the decoder
        cc0, hh0 = cchh0
        x = self.embedding(x)
        cc, hh = [], []
        for decoder_layer, c0, h0 in zip(self.decoder_layers, cc0, hh0):
            x, (c, h) = decoder_layer(x, encoder_outputs, (c0, h0))
            cc.append(c)
            hh.append(h)
        x = self.out_projection(x)
        return x, (cc, hh)

In [36]:
# Define synthetic training data
torch.manual_seed(42)
src_vocab_size = 20
tgt_vocab_size = 20
src_seq_length = 10
tgt_seq_length = 12
batch_size = 16

src_data = torch.randint(0, src_vocab_size, (batch_size, src_seq_length))
tgt_data = torch.randint(0, tgt_vocab_size, (batch_size, tgt_seq_length))

# Initialize models, loss function, and optimizer
input_dim = src_vocab_size
output_dim = tgt_vocab_size
embed_dim = 64
num_layers = 2

encoder = Encoder(src_vocab_size, embed_dim, num_layers)
decoder = Decoder(tgt_vocab_size, embed_dim, embed_dim, num_layers)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)

In [40]:
# Training loop
epochs = 100
for epoch in range(epochs):
    encoder_outputs, (cc, hh) = encoder(src_data)
    loss = 0
    decoder_input = torch.zeros(batch_size, 1, dtype=torch.long)  # Start token

    for t in range(tgt_seq_length):
        output, (cc, hh) = decoder(decoder_input, encoder_outputs, (cc, hh))
        loss += criterion(output[:, 0, :], tgt_data[:, t])
        decoder_input = tgt_data[:, t:t+1]  # Teacher forcing

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}] - Loss: {loss.item():.4f}")

Epoch [10/100] - Loss: 27.5924
Epoch [20/100] - Loss: 18.9763
Epoch [30/100] - Loss: 10.2055
Epoch [40/100] - Loss: 3.9769
Epoch [50/100] - Loss: 1.5607
Epoch [60/100] - Loss: 0.7655
Epoch [70/100] - Loss: 0.4734
Epoch [80/100] - Loss: 0.3395
Epoch [90/100] - Loss: 0.2664
Epoch [100/100] - Loss: 0.2207


In [44]:
# Test the sequence-to-sequence model with new input
test_input = torch.randint(0, src_vocab_size, (1, src_seq_length))
with torch.no_grad():
    encoder_outputs, (cell, hidden) = encoder(test_input)
    decoder_input = torch.zeros(1, 1, dtype=torch.long)  # Start token
    output_sequence = []

    for _ in range(tgt_seq_length):
        output, (cell, hidden) = decoder(decoder_input, encoder_outputs, (cell, hidden))
        predicted = output.argmax(dim=-1)
        output_sequence.append(predicted.item())
        decoder_input = predicted

    print(f"Input: {test_input.tolist()}, Output: {output_sequence}")

Input: [[12, 5, 0, 16, 13, 1, 12, 15, 0, 16]], Output: [12, 15, 19, 13, 7, 6, 2, 19, 12, 16, 14, 12]
